# Pandas — Guía Práctica Completa

## ¿Qué es Pandas?

**Pandas** es la librería estándar para trabajar con datos tabulares en Python.
Su nombre viene de *Panel Data* (datos de panel en econometría).

Imagina que tienes una hoja de cálculo de Excel o una tabla de base de datos:
Pandas te permite leerla, limpiarla, transformarla y analizarla con código Python.

## ¿Por qué necesitamos Pandas si ya tenemos listas y diccionarios?

Piensa en este problema: tienes 1000 filas de datos de estudiantes guardados en
una lista de diccionarios:

In [1]:
# SIN Pandas: datos como lista de diccionarios
datos = [
    {'nombre': 'Ana',    'nota': 85, 'carrera': 'Ingeniería'},
    {'nombre': 'Luis',   'nota': 72, 'carrera': 'Ciencias'},
    {'nombre': 'María',  'nota': 91, 'carrera': 'Ingeniería'},
    {'nombre': 'Carlos', 'nota': 60, 'carrera': 'Matemáticas'},
    {'nombre': 'Sofía',  'nota': 88, 'carrera': 'Ciencias'},
]

# Para calcular el promedio necesitas un ciclo:
suma = 0
for est in datos:
    suma += est['nota']
promedio_sin_pandas = suma / len(datos)
print(f'Promedio (sin Pandas): {promedio_sin_pandas}')

# Para filtrar estudiantes de Ingeniería también necesitas ciclo:
ingenieria = []
for est in datos:
    if est['carrera'] == 'Ingeniería':
        ingenieria.append(est['nombre'])
print(f'Ingeniería (sin Pandas): {ingenieria}')

Promedio (sin Pandas): 79.2
Ingeniería (sin Pandas): ['Ana', 'María']


In [2]:
import pandas as pd
import numpy as np

# CON Pandas: mucho más conciso
df = pd.DataFrame(datos)

print(f'Promedio (con Pandas): {df["nota"].mean()}')
print(f'Ingeniería (con Pandas): {df[df["carrera"]=="Ingeniería"]["nombre"].tolist()}')

print(f'\nPandas: {pd.__version__}  |  NumPy: {np.__version__}')

Promedio (con Pandas): 79.2
Ingeniería (con Pandas): ['Ana', 'María']

Pandas: 3.0.0  |  NumPy: 2.4.2


## ¿Cuándo usar Pandas?

| Tarea | ¿Pandas? |
|---|---|
| Leer archivos CSV, Excel, JSON | ✅ Ideal |
| Filtrar/agrupar/resumir datos tabulares | ✅ Ideal |
| Limpiar datos con NaN | ✅ Ideal |
| Series de tiempo | ✅ Ideal |
| Solo cálculos numéricos puros | ❌ Usa NumPy |
| Un solo array homogéneo | ❌ Usa NumPy |

---
## 1. La Series — una columna con etiquetas

Una **Series** es un array 1D con un **índice** (etiqueta para cada elemento).
Es como un diccionario ordenado que soporta operaciones matemáticas.

In [3]:
# Crear Series desde una lista — índice automático (0, 1, 2, ...)
temps = pd.Series([22.5, 23.1, 21.8, 24.0, 22.9])
print('Series con índice numérico:')
print(temps)
print(f'\nTipo Python:  {type(temps)}')
print(f'Tipo valores: {temps.dtype}')

Series con índice numérico:
0    22.5
1    23.1
2    21.8
3    24.0
4    22.9
dtype: float64

Tipo Python:  <class 'pandas.Series'>
Tipo valores: float64


In [4]:
# Crear Series con índice personalizado — como un diccionario ordenado
ventas = pd.Series(
    [1500, 2300, 1800, 2100, 1950],
    index=['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes'],
    name='Ventas_Q'
)
print('Ventas de la semana:')
print(ventas)

# Acceder por etiqueta (como diccionario)
print(f'\nVentas del martes: Q{ventas["Martes"]:,}')
# Acceder por posición (como lista)
print(f'Primer día:        Q{ventas.iloc[0]:,}')
print(f'Último día:        Q{ventas.iloc[-1]:,}')

Ventas de la semana:
Lunes        1500
Martes       2300
Miércoles    1800
Jueves       2100
Viernes      1950
Name: Ventas_Q, dtype: int64

Ventas del martes: Q2,300
Primer día:        Q1,500
Último día:        Q1,950


In [5]:
# Las Series soportan operaciones matemáticas (como NumPy)
iva = ventas * 0.12
total = ventas + iva

print('Ventas  →  IVA  →  Total con IVA')
for dia in ventas.index:
    print(f'  {dia:<12}: Q{ventas[dia]:>7,.0f} + Q{iva[dia]:>6,.0f} = Q{total[dia]:>8,.0f}')

Ventas  →  IVA  →  Total con IVA
  Lunes       : Q  1,500 + Q   180 = Q   1,680
  Martes      : Q  2,300 + Q   276 = Q   2,576
  Miércoles   : Q  1,800 + Q   216 = Q   2,016
  Jueves      : Q  2,100 + Q   252 = Q   2,352
  Viernes     : Q  1,950 + Q   234 = Q   2,184


In [6]:
# Filtrado en Series (igual que NumPy)
print('Días con ventas mayores a Q2000:')
print(ventas[ventas > 2000])

print(f'\nTotal semana:   Q{ventas.sum():,.0f}')
print(f'Mejor día:      {ventas.idxmax()} (Q{ventas.max():,.0f})')
print(f'Peor día:       {ventas.idxmin()} (Q{ventas.min():,.0f})')

Días con ventas mayores a Q2000:
Martes    2300
Jueves    2100
Name: Ventas_Q, dtype: int64

Total semana:   Q9,650
Mejor día:      Martes (Q2,300)
Peor día:       Lunes (Q1,500)


---
## 2. El DataFrame — una tabla completa

Un **DataFrame** es una tabla con filas y columnas etiquetadas.
Cada columna es una Series que comparte el mismo índice.

### 2.1 Crear desde un diccionario

In [7]:
# Cada clave del diccionario → una columna
# Cada valor (lista) → los datos de esa columna
df = pd.DataFrame({
    'nombre':   ['Ana', 'Luis', 'María', 'Carlos', 'Sofía', 'Pedro'],
    'edad':     [23, 31, 28, 25, 22, 35],
    'carrera':  ['Ingeniería', 'TI', 'Ingeniería', 'Ciencias', 'Ciencias', 'TI'],
    'salario':  [8500, 12000, 9800, 7500, 11200, 13500],
    'años_exp': [1, 7, 4, 2, 5, 10]
})

print(f'Forma del DataFrame: {df.shape}  ({df.shape[0]} filas × {df.shape[1]} columnas)')
df

Forma del DataFrame: (6, 5)  (6 filas × 5 columnas)


,nombre,edad,carrera,salario,años_exp
0,Ana,23,Ingeniería,8500,1
1,Luis,31,TI,12000,7
2,María,28,Ingeniería,9800,4
3,Carlos,25,Ciencias,7500,2
4,Sofía,22,Ciencias,11200,5
5,Pedro,35,TI,13500,10


In [8]:
# .info() muestra el resumen más importante del DataFrame
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   nombre    6 non-null      str  
 1   edad      6 non-null      int64
 2   carrera   6 non-null      str  
 3   salario   6 non-null      int64
 4   años_exp  6 non-null      int64
dtypes: int64(3), str(2)
memory usage: 372.0 bytes


In [9]:
# .describe() muestra estadísticas de columnas numéricas
# count, mean, std, min, 25%, 50%, 75%, max
df.describe().round(1)

,edad,salario,años_exp
count,6.0,6.0,6.0
mean,27.3,10416.7,4.8
std,5.0,2244.5,3.3
min,22.0,7500.0,1.0
25%,23.5,8825.0,2.5
50%,26.5,10500.0,4.5
75%,30.2,11800.0,6.5
max,35.0,13500.0,10.0


In [10]:
# .head(n) y .tail(n): primeras/últimas n filas
print('Primeras 3 filas:')
print(df.head(3))
print('\nÚltimas 2 filas:')
print(df.tail(2))

Primeras 3 filas:
  nombre  edad     carrera  salario  años_exp
0    Ana    23  Ingeniería     8500         1
1   Luis    31          TI    12000         7
2  María    28  Ingeniería     9800         4

Últimas 2 filas:
  nombre  edad   carrera  salario  años_exp
4  Sofía    22  Ciencias    11200         5
5  Pedro    35        TI    13500        10


---
## 3. Selección de Datos

Hay tres formas principales de seleccionar datos en un DataFrame:

| Método | ¿Qué selecciona? | Ejemplo |
|---|---|---|
| `df['col']` | Una columna | `df['nombre']` |
| `df.loc[fila, col]` | Por **etiqueta** | `df.loc[0, 'nombre']` |
| `df.iloc[fila, col]` | Por **posición entera** | `df.iloc[0, 0]` |

### 3.1 Seleccionar columnas

In [11]:
# Una columna → devuelve una Series
nombres = df['nombre']
print('Una columna (Series):')
print(nombres)
print(type(nombres))

Una columna (Series):
0       Ana
1      Luis
2     María
3    Carlos
4     Sofía
5     Pedro
Name: nombre, dtype: str
<class 'pandas.Series'>


In [12]:
# Varias columnas → devuelve DataFrame
# NOTA: doble corchete [[ ]] porque pasas una LISTA de columnas
print('Varias columnas (DataFrame):')
df[['nombre', 'carrera', 'salario']]

Varias columnas (DataFrame):


,nombre,carrera,salario
0,Ana,Ingeniería,8500
1,Luis,TI,12000
2,María,Ingeniería,9800
3,Carlos,Ciencias,7500
4,Sofía,Ciencias,11200
5,Pedro,TI,13500


### 3.2 Seleccionar filas con `iloc` y `loc`

- `iloc` usa **posiciones enteras** (0, 1, 2...) — como listas de Python
- `loc`  usa **etiquetas del índice** — si el índice es numérico, puede confundirse con iloc

In [13]:
print('Primera fila (iloc[0]):')
print(df.iloc[0])

print('\nÚltima fila (iloc[-1]):')
print(df.iloc[-1])

print('\nFilas 1, 2, 3 (iloc[1:4]):')
print(df.iloc[1:4])

Primera fila (iloc[0]):
nombre             Ana
edad                23
carrera     Ingeniería
salario           8500
años_exp             1
Name: 0, dtype: object

Última fila (iloc[-1]):
nombre      Pedro
edad           35
carrera        TI
salario     13500
años_exp       10
Name: 5, dtype: object

Filas 1, 2, 3 (iloc[1:4]):
   nombre  edad     carrera  salario  años_exp
1    Luis    31          TI    12000         7
2   María    28  Ingeniería     9800         4
3  Carlos    25    Ciencias     7500         2


In [14]:
# loc con filas Y columnas específicas
print('Fila 0, columna nombre (loc[0, "nombre"]):')
print(df.loc[0, 'nombre'])

print('\nFilas 0 a 2, columnas nombre y salario:')
print(df.loc[0:2, ['nombre', 'salario']])  # NOTA: loc incluye el extremo derecho (2)

Fila 0, columna nombre (loc[0, "nombre"]):
Ana

Filas 0 a 2, columnas nombre y salario:
  nombre  salario
0    Ana     8500
1   Luis    12000
2  María     9800


In [15]:
# Diferencia importante: loc incluye el límite derecho, iloc no
print('df.iloc[0:3] → filas 0, 1, 2 (el 3 NO incluido):')
print(df.iloc[0:3][['nombre']])

print('\ndf.loc[0:3] → filas 0, 1, 2, 3 (el 3 SÍ incluido):')
print(df.loc[0:3][['nombre']])

df.iloc[0:3] → filas 0, 1, 2 (el 3 NO incluido):
  nombre
0    Ana
1   Luis
2  María

df.loc[0:3] → filas 0, 1, 2, 3 (el 3 SÍ incluido):
   nombre
0     Ana
1    Luis
2   María
3  Carlos


---
## 4. Filtrado con Condiciones

El filtrado es como un ciclo `for` con `if`, pero mucho más conciso.
El principio es el mismo que en NumPy: crear una máscara booleana.

In [16]:
# Filtrado simple: empleados con salario mayor a 10,000
# Equivalente al ciclo:
# resultado = []
# for i, row in df.iterrows():
#     if row['salario'] > 10000:
#         resultado.append(row)

bien_pagados = df[df['salario'] > 10_000]
print('Empleados con salario > Q10,000:')
print(bien_pagados)

Empleados con salario > Q10,000:
  nombre  edad   carrera  salario  años_exp
1   Luis    31        TI    12000         7
4  Sofía    22  Ciencias    11200         5
5  Pedro    35        TI    13500        10


In [17]:
# Múltiples condiciones
# & = AND (ambas deben ser True)
# | = OR  (al menos una debe ser True)
# IMPORTANTE: cada condición entre paréntesis

print('Ingeniería Y salario > 8000:')
print(df[(df['carrera'] == 'Ingeniería') & (df['salario'] > 8000)])

print('\nTI O más de 5 años de experiencia:')
print(df[(df['carrera'] == 'TI') | (df['años_exp'] > 5)])

print('\n.isin() — equivalente a varios OR en una sola línea:')
print(df[df['carrera'].isin(['TI', 'Ciencias'])])

Ingeniería Y salario > 8000:
  nombre  edad     carrera  salario  años_exp
0    Ana    23  Ingeniería     8500         1
2  María    28  Ingeniería     9800         4

TI O más de 5 años de experiencia:
  nombre  edad carrera  salario  años_exp
1   Luis    31      TI    12000         7
5  Pedro    35      TI    13500        10

.isin() — equivalente a varios OR en una sola línea:
   nombre  edad   carrera  salario  años_exp
1    Luis    31        TI    12000         7
3  Carlos    25  Ciencias     7500         2
4   Sofía    22  Ciencias    11200         5
5   Pedro    35        TI    13500        10


In [18]:
# .query(): sintaxis tipo SQL (más legible para condiciones complejas)
print('Con query: salario > 9000 y experiencia >= 3:')
print(df.query('salario > 9000 and años_exp >= 3'))

Con query: salario > 9000 y experiencia >= 3:
  nombre  edad     carrera  salario  años_exp
1   Luis    31          TI    12000         7
2  María    28  Ingeniería     9800         4
4  Sofía    22    Ciencias    11200         5
5  Pedro    35          TI    13500        10


In [19]:
# Caso de uso: análisis con ciclo for sobre grupos filtrados
carreras_unicas = df['carrera'].unique()
print('Análisis por carrera:')
print(f'{"Carrera":<15} {"N":>3} {"Sal.Prom":>10} {"Sal.Max":>9}')
print('-' * 42)

for carrera in sorted(carreras_unicas):
    subset = df[df['carrera'] == carrera]
    n      = len(subset)
    prom   = subset['salario'].mean()
    maximo = subset['salario'].max()
    print(f'{carrera:<15} {n:>3} {prom:>10,.0f} {maximo:>9,.0f}')

Análisis por carrera:
Carrera           N   Sal.Prom   Sal.Max
------------------------------------------
Ciencias          2      9,350    11,200
Ingeniería        2      9,150     9,800
TI                2     12,750    13,500


---
## 5. Crear y Transformar Columnas

In [20]:
df2 = df.copy()  # trabajar en una copia para no modificar el original

# Crear columna simple: operación sobre columnas existentes
df2['salario_anual'] = df2['salario'] * 12
df2['bono_anual']    = df2['salario'] * 0.10 * 12  # 10% de bono

print('Salario mensual y anual:')
print(df2[['nombre', 'salario', 'salario_anual', 'bono_anual']])

Salario mensual y anual:
   nombre  salario  salario_anual  bono_anual
0     Ana     8500         102000     10200.0
1    Luis    12000         144000     14400.0
2   María     9800         117600     11760.0
3  Carlos     7500          90000      9000.0
4   Sofía    11200         134400     13440.0
5   Pedro    13500         162000     16200.0


In [21]:
# .apply(): aplicar una función propia a cada elemento
# Es como un for en una línea
def categoria_salarial(salario):
    """Clasifica el salario en una categoría."""
    if salario < 9000:
        return 'Junior'
    elif salario < 12000:
        return 'Semi-Senior'
    else:
        return 'Senior'

df2['nivel'] = df2['salario'].apply(categoria_salarial)

print('Empleados con nivel asignado:')
print(df2[['nombre', 'salario', 'nivel']])

Empleados con nivel asignado:
   nombre  salario        nivel
0     Ana     8500       Junior
1    Luis    12000       Senior
2   María     9800  Semi-Senior
3  Carlos     7500       Junior
4   Sofía    11200  Semi-Senior
5   Pedro    13500       Senior


In [22]:
# pd.cut(): dividir datos numéricos en rangos (bins)
# Más robusto que if/elif cuando hay muchas categorías
df2['rango_edad'] = pd.cut(
    df2['edad'],
    bins=[0, 25, 30, 100],
    labels=['Joven (≤25)', 'Adulto (26-30)', 'Maduro (>30)']
)

print('Conteo por rango de edad:')
print(df2['rango_edad'].value_counts())

Conteo por rango de edad:
rango_edad
Joven (≤25)       3
Maduro (>30)      2
Adulto (26-30)    1
Name: count, dtype: int64


In [23]:
# Operaciones con strings: .str accessor
# Evita tener que hacer ciclos para manipular texto
df2['nombre_upper'] = df2['nombre'].str.upper()
df2['inicial']      = df2['nombre'].str[0]     # primer carácter
df2['carrera_corta']= df2['carrera'].str[:3]   # primeras 3 letras

print(df2[['nombre', 'nombre_upper', 'inicial', 'carrera', 'carrera_corta']])

   nombre nombre_upper inicial     carrera carrera_corta
0     Ana          ANA       A  Ingeniería           Ing
1    Luis         LUIS       L          TI            TI
2   María        MARÍA       M  Ingeniería           Ing
3  Carlos       CARLOS       C    Ciencias           Cie
4   Sofía        SOFÍA       S    Ciencias           Cie
5   Pedro        PEDRO       P          TI            TI


---
## 6. Valores Faltantes (NaN)

`NaN` significa "Not a Number". En Pandas representa datos faltantes.

**¿Por qué aparecen NaN?**
- El encuestado no respondió una pregunta
- El sensor falló y no registró el valor
- Al combinar tablas, una fila no tiene correspondencia
- Error al cargar el CSV

**Regla importante:** No puedes ignorarlos — afectan todas las operaciones.

In [24]:
datos_raw = pd.DataFrame({
    'nombre':  ['Ana', 'Luis', 'María', 'Carlos', 'Sofía', 'Pedro'],
    'nota1':   [85.0, None, 78.0, 92.0, None, 70.0],
    'nota2':   [90.0, 75.0, None, 88.0, 82.0, None],
    'ciudad':  ['Guatemala', None, 'Cobán', 'Antigua', 'Guatemala', None]
})
print('Datos con NaN:')
datos_raw

Datos con NaN:


,nombre,nota1,nota2,ciudad
0,Ana,85.0,90.0,Guatemala
1,Luis,NaN,75.0,NaN
2,María,78.0,NaN,Cobán
3,Carlos,92.0,88.0,Antigua
4,Sofía,NaN,82.0,Guatemala
5,Pedro,70.0,NaN,NaN


In [25]:
# Detectar nulos
print('Cantidad de nulos por columna:')
print(datos_raw.isnull().sum())
print(f'\nTotal de nulos: {datos_raw.isnull().sum().sum()}')

# Ver qué filas tienen al menos un nulo
print('\nFilas con algún nulo:')
print(datos_raw[datos_raw.isnull().any(axis=1)])

Cantidad de nulos por columna:
nombre    0
nota1     2
nota2     2
ciudad    2
dtype: int64

Total de nulos: 6

Filas con algún nulo:
  nombre  nota1  nota2     ciudad
1   Luis    NaN   75.0        NaN
2  María   78.0    NaN      Cobán
4  Sofía    NaN   82.0  Guatemala
5  Pedro   70.0    NaN        NaN


In [26]:
# Estrategia 1: ELIMINAR filas con NaN (solo si tienes muchos datos)
df_sin_nulos = datos_raw.dropna()
print(f'Sin dropna: {len(datos_raw)} filas')
print(f'Con dropna: {len(df_sin_nulos)} filas')
print(df_sin_nulos)

Sin dropna: 6 filas
Con dropna: 2 filas
   nombre  nota1  nota2     ciudad
0     Ana   85.0   90.0  Guatemala
3  Carlos   92.0   88.0    Antigua


In [27]:
# Estrategia 2: IMPUTAR — rellenar con un valor calculado
# Para notas: usar la mediana (resistente a outliers)
# Para ciudad: usar el valor más frecuente (moda)

df_imputado = datos_raw.copy()

for col in ['nota1', 'nota2']:
    mediana = df_imputado[col].median()
    nulos_antes = df_imputado[col].isnull().sum()
    df_imputado[col] = df_imputado[col].fillna(mediana)
    print(f'{col}: {nulos_antes} NaN imputados con mediana={mediana}')

moda_ciudad = df_imputado['ciudad'].mode()[0]
df_imputado['ciudad'] = df_imputado['ciudad'].fillna(moda_ciudad)
print(f'ciudad: NaN imputados con moda="{moda_ciudad}"')

print('\nDataFrame imputado:')
df_imputado

nota1: 2 NaN imputados con mediana=81.5
nota2: 2 NaN imputados con mediana=85.0
ciudad: NaN imputados con moda="Guatemala"

DataFrame imputado:


,nombre,nota1,nota2,ciudad
0,Ana,85.0,90.0,Guatemala
1,Luis,81.5,75.0,Guatemala
2,María,78.0,85.0,Cobán
3,Carlos,92.0,88.0,Antigua
4,Sofía,81.5,82.0,Guatemala
5,Pedro,70.0,85.0,Guatemala


In [28]:
# Estrategia 3: relleno hacia adelante (ffill) y hacia atrás (bfill)
# Muy común en series de tiempo donde un NaN = "mismo valor que antes"
serie_temp = pd.Series([20.5, None, None, 22.0, None, 23.5])
print('Original:         ', serie_temp.tolist())
print('ffill (→ adelante):', serie_temp.ffill().tolist())
print('bfill (← atrás):  ', serie_temp.bfill().tolist())

Original:          [20.5, nan, nan, 22.0, nan, 23.5]
ffill (→ adelante): [20.5, 20.5, 20.5, 22.0, 22.0, 23.5]
bfill (← atrás):   [20.5, 22.0, 22.0, 22.0, 23.5, 23.5]


---
## 7. Ordenamiento y Clasificación

In [29]:
# Ordenar por una columna
print('Por salario (mayor a menor):')
print(df.sort_values('salario', ascending=False).to_string(index=False))

# Ordenar por múltiples columnas
print('\nPor carrera (A-Z) y luego por salario (mayor a menor):')
print(df.sort_values(['carrera', 'salario'], ascending=[True, False]).to_string(index=False))

Por salario (mayor a menor):
nombre  edad    carrera  salario  años_exp
 Pedro    35         TI    13500        10
  Luis    31         TI    12000         7
 Sofía    22   Ciencias    11200         5
 María    28 Ingeniería     9800         4
   Ana    23 Ingeniería     8500         1
Carlos    25   Ciencias     7500         2

Por carrera (A-Z) y luego por salario (mayor a menor):
nombre  edad    carrera  salario  años_exp
 Sofía    22   Ciencias    11200         5
Carlos    25   Ciencias     7500         2
 María    28 Ingeniería     9800         4
   Ana    23 Ingeniería     8500         1
 Pedro    35         TI    13500        10
  Luis    31         TI    12000         7


In [30]:
# .rank(): asignar un ranking a cada valor
# Caso de uso: determinar el puesto de cada vendedor
df3 = df.copy()
df3['ranking_salario'] = df3['salario'].rank(ascending=False).astype(int)
print(df3[['nombre', 'salario', 'ranking_salario']].sort_values('ranking_salario'))

   nombre  salario  ranking_salario
5   Pedro    13500                1
1    Luis    12000                2
4   Sofía    11200                3
2   María     9800                4
0     Ana     8500                5
3  Carlos     7500                6


---
## 8. GroupBy — Agrupar y Resumir

`groupby` es una de las operaciones más poderosas de Pandas.
Funciona en tres pasos: **Split → Apply → Combine**

1. **Split**: divide el DataFrame en grupos según una columna
2. **Apply**: aplica una función a cada grupo
3. **Combine**: junta los resultados

Es el equivalente de un `GROUP BY` en SQL.

In [31]:
# Dataset de ventas más rico
ventas = pd.DataFrame({
    'vendedor': ['Ana','Ana','Ana','Luis','Luis','Luis','María','María','María'],
    'mes':      ['Ene','Feb','Mar','Ene','Feb','Mar','Ene','Feb','Mar'],
    'producto': ['A','B','A','A','A','B','B','B','A'],
    'cantidad': [10, 15, 12, 8,  20, 14, 18, 9,  11],
    'precio':   [50, 30, 50, 50, 50, 30, 30, 30, 50]
})
ventas['total'] = ventas['cantidad'] * ventas['precio']
print('Dataset de ventas:')
print(ventas)

Dataset de ventas:
  vendedor  mes producto  cantidad  precio  total
0      Ana  Ene        A        10      50    500
1      Ana  Feb        B        15      30    450
2      Ana  Mar        A        12      50    600
3     Luis  Ene        A         8      50    400
4     Luis  Feb        A        20      50   1000
5     Luis  Mar        B        14      30    420
6    María  Ene        B        18      30    540
7    María  Feb        B         9      30    270
8    María  Mar        A        11      50    550


In [32]:
# groupby simple: total de ventas por vendedor
# Equivalente al ciclo:
# vendedores = {}
# for _, row in ventas.iterrows():
#     if row['vendedor'] not in vendedores:
#         vendedores[row['vendedor']] = 0
#     vendedores[row['vendedor']] += row['total']

por_vendedor = ventas.groupby('vendedor')['total'].sum()
print('Total de ventas por vendedor:')
print(por_vendedor)

Total de ventas por vendedor:
vendedor
Ana      1550
Luis     1820
María    1360
Name: total, dtype: int64


In [33]:
# .agg(): calcular varios estadísticos a la vez
resumen = ventas.groupby('vendedor').agg(
    total_ventas  = ('total',    'sum'),
    venta_promedio= ('total',    'mean'),
    mejor_venta   = ('total',    'max'),
    num_ventas    = ('total',    'count')
).round(1)

print('Resumen por vendedor:')
resumen

Resumen por vendedor:


,total_ventas,venta_promedio,mejor_venta,num_ventas
vendedor,,,,
Ana,1550,516.7,600,3
Luis,1820,606.7,1000,3
María,1360,453.3,550,3


In [34]:
# Iterar sobre grupos con un ciclo for
# Útil cuando necesitas lógica más compleja por grupo
print('Análisis detallado por vendedor:')
print('=' * 45)

for nombre, grupo in ventas.groupby('vendedor'):
    total  = grupo['total'].sum()
    mejor  = grupo.loc[grupo['total'].idxmax()]
    print(f'\nVendedor: {nombre}')
    print(f'  Total vendido: Q{total:,.0f}')
    print(f'  Mejor venta:   {mejor["mes"]} producto {mejor["producto"]} (Q{mejor["total"]})')
    print(f'  Meses activos: {grupo["mes"].nunique()}')

Análisis detallado por vendedor:

Vendedor: Ana
  Total vendido: Q1,550
  Mejor venta:   Mar producto A (Q600)
  Meses activos: 3

Vendedor: Luis
  Total vendido: Q1,820
  Mejor venta:   Feb producto A (Q1000)
  Meses activos: 3

Vendedor: María
  Total vendido: Q1,360
  Mejor venta:   Mar producto A (Q550)
  Meses activos: 3


---
## 9. Pivot Tables

Una tabla dinámica (pivot table) reorganiza los datos para comparar dos dimensiones.
Es como una tabla de doble entrada.

In [35]:
# Pivot: filas = vendedor, columnas = mes, valores = total de ventas
tabla = pd.pivot_table(
    ventas,
    values='total',
    index='vendedor',
    columns='mes',
    aggfunc='sum',
    margins=True,           # añade totales
    margins_name='TOTAL'
)
print('Ventas por vendedor y mes:')
tabla

Ventas por vendedor y mes:


mes,Ene,Feb,Mar,TOTAL
vendedor,,,,
Ana,500,450,600,1550
Luis,400,1000,420,1820
María,540,270,550,1360
TOTAL,1440,1720,1570,4730


---
## 10. Merge — Combinar Tablas

Combina dos DataFrames usando una columna en común, como un JOIN en SQL.

```
INNER JOIN: solo filas que existen en AMBAS tablas
LEFT JOIN:  todas las filas de la tabla izquierda + lo que haya en la derecha
RIGHT JOIN: todas las filas de la tabla derecha + lo que haya en la izquierda
OUTER JOIN: todas las filas de ambas tablas
```

In [36]:
# Tabla de estudiantes
estudiantes = pd.DataFrame({
    'id':        [1, 2, 3, 4, 5],
    'nombre':    ['Ana', 'Luis', 'María', 'Carlos', 'Sofía'],
    'carrera_id': [10, 20, 10, 30, 20]
})

# Tabla de carreras
carreras = pd.DataFrame({
    'carrera_id': [10, 20, 30],
    'carrera':    ['Ingeniería', 'Ciencias', 'Matemáticas'],
    'facultad':   ['FIUSAC', 'ECFM', 'ECFM']
})

print('Estudiantes:')
print(estudiantes)
print('\nCarreras:')
print(carreras)

Estudiantes:
   id  nombre  carrera_id
0   1     Ana          10
1   2    Luis          20
2   3   María          10
3   4  Carlos          30
4   5   Sofía          20

Carreras:
   carrera_id      carrera facultad
0          10   Ingeniería   FIUSAC
1          20     Ciencias     ECFM
2          30  Matemáticas     ECFM


In [37]:
# INNER JOIN: cada estudiante con el nombre de su carrera
resultado = pd.merge(estudiantes, carreras, on='carrera_id', how='inner')
print('Resultado del INNER JOIN:')
print(resultado)

Resultado del INNER JOIN:
   id  nombre  carrera_id      carrera facultad
0   1     Ana          10   Ingeniería   FIUSAC
1   2    Luis          20     Ciencias     ECFM
2   3   María          10   Ingeniería   FIUSAC
3   4  Carlos          30  Matemáticas     ECFM
4   5   Sofía          20     Ciencias     ECFM


In [38]:
# Tabla de notas (no tiene todos los estudiantes)
notas_tabla = pd.DataFrame({
    'id':   [1, 2, 4],   # Carlos (id=3) y Sofía (id=5) no tienen nota aún
    'nota': [85, 72, 91]
})

print('LEFT JOIN (todos los estudiantes, hayan o no entregado):')
resultado_left = pd.merge(estudiantes, notas_tabla, on='id', how='left')
print(resultado_left)
print('\nNota: NaN = estudiante sin calificación registrada')

LEFT JOIN (todos los estudiantes, hayan o no entregado):
   id  nombre  carrera_id  nota
0   1     Ana          10  85.0
1   2    Luis          20  72.0
2   3   María          10   NaN
3   4  Carlos          30  91.0
4   5   Sofía          20   NaN

Nota: NaN = estudiante sin calificación registrada


---
## 11. Fechas y Series de Tiempo

Pandas tiene soporte nativo para fechas, lo que lo hace ideal para analizar
datos con componente temporal.

In [39]:
# Crear un rango de fechas
fechas = pd.date_range(start='2024-01-01', periods=10, freq='D')
print('10 días a partir del 1 de enero 2024:')
print(fechas)

10 días a partir del 1 de enero 2024:
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10'],
              dtype='datetime64[us]', freq='D')


In [40]:
# Crear Series de tiempo y analizarla
rng = np.random.default_rng(42)
ventas_diarias = pd.Series(
    rng.integers(500, 2000, size=365),
    index=pd.date_range('2024-01-01', periods=365, freq='D'),
    name='ventas'
)

print('Ventas de enero 2024 (primeras 5):')
print(ventas_diarias['2024-01'].head())

# Resamplear: agrupar por semana, mes, etc.
ventas_mensuales = ventas_diarias.resample('ME').sum()
print('\nVentas totales por mes:')
for fecha, total in ventas_mensuales.items():
    mes_nombre = fecha.strftime('%B')  # nombre del mes
    print(f'  {mes_nombre}: Q{total:>7,}')

Ventas de enero 2024 (primeras 5):
2024-01-01     633
2024-01-02    1660
2024-01-03    1481
2024-01-04    1158
2024-01-05    1149
Freq: D, Name: ventas, dtype: int64

Ventas totales por mes:
  January: Q 40,732
  February: Q 36,512
  March: Q 39,840
  April: Q 37,627
  May: Q 38,835
  June: Q 37,563
  July: Q 39,560
  August: Q 37,119
  September: Q 35,145
  October: Q 37,999
  November: Q 35,804
  December: Q 40,287


In [41]:
# Extraer componentes de una fecha con .dt
df_f = pd.DataFrame({
    'evento': ['Inicio', 'Primer parcial', 'Vacaciones', 'Segundo parcial', 'Final'],
    'fecha':  pd.to_datetime(['2024-01-15', '2024-03-01', '2024-04-05',
                               '2024-05-10', '2024-06-20'])
})

df_f['mes']        = df_f['fecha'].dt.month
df_f['nombre_mes'] = df_f['fecha'].dt.month_name()
df_f['dia_semana'] = df_f['fecha'].dt.day_name()
df_f['semana_año'] = df_f['fecha'].dt.isocalendar().week

print('Calendario de eventos del curso:')
print(df_f.to_string(index=False))

Calendario de eventos del curso:
         evento      fecha  mes nombre_mes dia_semana  semana_año
         Inicio 2024-01-15    1    January     Monday           3
 Primer parcial 2024-03-01    3      March     Friday           9
     Vacaciones 2024-04-05    4      April     Friday          14
Segundo parcial 2024-05-10    5        May     Friday          19
          Final 2024-06-20    6       June   Thursday          25


---
## 🏋️ Ejercicio 1: Análisis de Notas del Curso

Tienes el siguiente DataFrame con las notas de 8 estudiantes en 3 exámenes.

**Tareas:**
1. Calcula el promedio de cada estudiante
2. Clasifica el estado: Aprobado (promedio ≥ 61), Reprobado
3. Imprime quién fue el mejor y el peor estudiante
4. Cuenta cuántos aprobaron y reprobaron por carrera

In [42]:
# Datos del ejercicio
df_curso = pd.DataFrame({
    'nombre':  ['Ana','Luis','María','Carlos','Sofía','Pedro','Elena','Marcos'],
    'carrera': ['Ing.','Cien.','Ing.','Mat.','Cien.','Ing.','Mat.','Cien.'],
    'ex1':     [75, 45, 88, 60, 92, 55, 70, 80],
    'ex2':     [80, 50, 85, 65, 88, 60, 72, 75],
    'ex3':     [70, 40, 91, 58, 95, 62, 68, 85]
})

# 1. Calcular promedio
df_curso['promedio'] = df_curso[['ex1','ex2','ex3']].mean(axis=1).round(1)

# 2. Clasificar estado
df_curso['estado'] = df_curso['promedio'].apply(
    lambda p: 'Aprobado' if p >= 61 else 'Reprobado'
)

print('DataFrame con promedio y estado:')
print(df_curso.to_string(index=False))

# 3. Mejor y peor
mejor = df_curso.loc[df_curso['promedio'].idxmax()]
peor  = df_curso.loc[df_curso['promedio'].idxmin()]
print(f'\nMejor estudiante: {mejor["nombre"]} ({mejor["promedio"]})')
print(f'Peor estudiante:  {peor["nombre"]} ({peor["promedio"]})')

# 4. Conteo por carrera y estado
print('\nAprobados y reprobados por carrera:')
resumen = df_curso.groupby(['carrera','estado']).size().unstack(fill_value=0)
print(resumen)

DataFrame con promedio y estado:
nombre carrera  ex1  ex2  ex3  promedio    estado
   Ana    Ing.   75   80   70      75.0  Aprobado
  Luis   Cien.   45   50   40      45.0 Reprobado
 María    Ing.   88   85   91      88.0  Aprobado
Carlos    Mat.   60   65   58      61.0  Aprobado
 Sofía   Cien.   92   88   95      91.7  Aprobado
 Pedro    Ing.   55   60   62      59.0 Reprobado
 Elena    Mat.   70   72   68      70.0  Aprobado
Marcos   Cien.   80   75   85      80.0  Aprobado

Mejor estudiante: Sofía (91.7)
Peor estudiante:  Luis (45.0)

Aprobados y reprobados por carrera:
estado   Aprobado  Reprobado
carrera                     
Cien.           2          1
Ing.            2          1
Mat.            2          0


---
## 🏋️ Ejercicio 2: Limpieza y Análisis de Datos Reales

El siguiente DataFrame simula datos "sucios" que podrías encontrar en la vida real.

**Tareas:**
1. Detecta todos los problemas (NaN, duplicados, valores fuera de rango)
2. Limpia el DataFrame
3. Calcula el resumen estadístico por ciudad

In [43]:
# DataFrame con datos "sucios"
df_sucio = pd.DataFrame({
    'nombre': ['Ana','Luis','Ana','María',None,'Carlos','Sofía','Luis'],
    'ciudad': ['Guatemala','Cobán','Guatemala','Antigua','Guatemala','Cobán',None,'Cobán'],
    'edad':   [23, 31, 23, 150, 28, 25, 22, 31],   # 150 es error
    'compra': [500, None, 500, 300, 800, None, 450, 600]
})
print('DataFrame sucio:')
print(df_sucio)

print(f'\nNulos: {df_sucio.isnull().sum().to_dict()}')
print(f'Duplicados: {df_sucio.duplicated().sum()}')

# Limpieza
df_limpio = df_sucio.copy()
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)  # eliminar duplicados
df_limpio['edad']   = df_limpio['edad'].clip(0, 100)            # limitar edad
df_limpio['compra'] = df_limpio['compra'].fillna(df_limpio['compra'].median())
df_limpio['ciudad'] = df_limpio['ciudad'].fillna('Desconocida')
df_limpio = df_limpio.dropna(subset=['nombre'])                 # si nombre es NaN, eliminar

print('\nDataFrame limpio:')
print(df_limpio)

print('\nResumen por ciudad:')
print(df_limpio.groupby('ciudad')['compra'].agg(['count','mean','sum']).round(1))

DataFrame sucio:
   nombre     ciudad  edad  compra
0     Ana  Guatemala    23   500.0
1    Luis      Cobán    31     NaN
2     Ana  Guatemala    23   500.0
3   María    Antigua   150   300.0
4     NaN  Guatemala    28   800.0
5  Carlos      Cobán    25     NaN
6   Sofía        NaN    22   450.0
7    Luis      Cobán    31   600.0

Nulos: {'nombre': 1, 'ciudad': 1, 'edad': 0, 'compra': 2}
Duplicados: 1

DataFrame limpio:
   nombre       ciudad  edad  compra
0     Ana    Guatemala    23   500.0
1    Luis        Cobán    31   500.0
2   María      Antigua   100   300.0
4  Carlos        Cobán    25   500.0
5   Sofía  Desconocida    22   450.0
6    Luis        Cobán    31   600.0

Resumen por ciudad:
             count   mean     sum
ciudad                           
Antigua          1  300.0   300.0
Cobán            3  533.3  1600.0
Desconocida      1  450.0   450.0
Guatemala        1  500.0   500.0
